# Make a PSF for a simple exoplanet

This tutorial is a Jupyter notebook, located at `docs/tutorials/exoplanet-psf.ipynb` in the project.

Read the notebook on this page, or download {download}`exoplanet-psf.ipynb <exoplanet-psf.ipynb>` to run it locally.

**Next step:** [Select phantoms and prepare particle inputs](../running-atris/prepare-simulation.md#optionally-select-phantoms).

Let's make `simulation.psf` for a hypothetical rocky planet with a **pure nitrogen atmosphere**. Choose its size and surface conditions, calculate the atmosphere, plot it, and save the file.

This is a teaching model: constant temperature, constant gravity, and pure N₂. The parameters are assumptions, not measurements of a known planet.

Run the cells from top to bottom using a Python kernel with NumPy and Matplotlib installed. The last cell writes `exoplanet-inputs/simulation.psf` relative to the notebook's working directory (and replaces it if it already exists).

## 1. Choose the planet

Use SI units for the calculation: metres, kilograms, kelvin, and pascals. Change these values and rerun the notebook to see how the atmosphere responds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

radius = 1.5 * 6.371e6     # m; 1.5 Earth radii
mass = 4.0 * 5.9722e24     # kg; 4 Earth masses
temperature = 300.0       # K
surface_pressure = 1e5   # Pa (1 bar)
top = 100e3              # m; atmosphere extends to 100 km
n_layers = 50

## 2. Calculate the atmosphere

For an isothermal atmosphere with constant gravity, pressure decreases with altitude $z$ as

$$
P(z)=P_0\exp\left(-\frac{z}{H}\right),\qquad
H=\frac{RT}{Mg},\qquad g=\frac{Gm}{r^2}.
$$

$P_0$ is surface pressure, $H$ is scale height, $T$ is temperature, $M$ is the
molar mass of nitrogen, and $m$ and $r$ are the planet's mass and radius.
The ideal-gas relation gives density:

$$
\rho(z)=\frac{P(z)M}{RT}.
$$

In the code, `surface_pressure`, `scale_height`, and `molar_mass` correspond to
$P_0$, $H$, and $M$.

NumPy calculates all layers at once. `edges` contains their boundaries; `altitude` contains their midpoints, where we sample pressure and density. Each AtRIS layer gets one constant set of values. Constant gravity is an approximation for an atmosphere thin compared with the planet's radius.

In [ ]:
G = 6.67430e-11       # gravitational constant, m³ kg⁻¹ s⁻²
R = 8.314462618       # gas constant, J mol⁻¹ K⁻¹
molar_mass = 0.0280134  # kg/mol, N₂

gravity = G * mass / radius**2
scale_height = R * temperature / (molar_mass * gravity)

edges = np.linspace(0, top, n_layers + 1)
altitude = (edges[:-1] + edges[1:]) / 2
pressure = surface_pressure * np.exp(-altitude / scale_height)
density = pressure * molar_mass / (R * temperature)  # kg/m³

print(f"Gravity: {gravity:.2f} m/s²")
print(f"Scale height: {scale_height / 1000:.2f} km")

## 3. Plot the profiles

These are the profiles we will put in the PSF. Pressure and density span many orders of magnitude, so we use logarithmic horizontal axes. Temperature is constant by assumption. The atmosphere's elemental nitrogen mass fraction is 1 everywhere.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4), sharey=True, layout="constrained")

axes[0].plot(np.full(n_layers, temperature), altitude / 1000)
axes[0].set_xlabel("Temperature [K]")
axes[0].set_ylabel("Altitude [km]")

axes[1].semilogx(pressure, altitude / 1000)
axes[1].set_xlabel("Pressure [Pa]")

axes[2].semilogx(density, altitude / 1000)
axes[2].set_xlabel("Density [kg/m³]")

for ax in axes:
    ax.grid(alpha=0.3)
axes[0].set_ylim(0, top / 1000)
plt.show()

## 4. Write `simulation.psf`

**The PSF (Planet Specification Format) describes the planet's geometry and material in each layer.** Each numeric row is one spherical shell: where it starts and ends, what it is made of, and its temperature, density, and pressure. It is the model input, not a simulation result. Particle energies and event counts are specified separately.

For this example, rows 0 and 1 describe the solid interior and thin surface layer. Rows 2 onward describe the atmosphere, with one row per altitude interval. The final row is an outer stopping shell. Radii are measured **from the planet's centre**, not above its surface.

### The material header

The first line of our file is:

```text
O533Si467 N
```

`O533Si467` defines the shared solid composition: oxygen and silicon with relative mass weights 533:467 (53.3% and 46.7%). `N` names the element in the last column of the numeric table. For gas rows, that column contains the nitrogen **mass fraction**. Pure N₂ therefore gives `1`, since all its mass is nitrogen; the PSF stores elemental composition rather than molecular abundances. Solid rows use the header's solid composition and have `0` in the unused nitrogen column.

### The first four rows

This is a rounded preview for the default parameters above. The actual file contains whitespace-separated numbers, without the Markdown column labels. If you change the parameters, the exported values will change too.

| Inner radius (km) | Outer radius (km) | φ start (rad) | φ end (rad) | θ start (rad) | θ end (rad) | ID | T (K) | Density (g/cm³) | Pressure (Pa) | Solid | N mass fraction |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 9556.48 | 0 | 6.28319 | 0 | 3.14159 | 0 | 300 | 2.7 | 100000 | 1 | 0 |
| 9556.48 | 9556.50 | 0 | 6.28319 | 0 | 3.14159 | 1 | 300 | 2.7 | 100000 | 1 | 0 |
| 9556.50 | 9558.50 | 0 | 6.28319 | 0 | 3.14159 | 2 | 300 | 0.00092312 | 82195.4 | 0 | 1 |
| 9558.50 | 9560.50 | 0 | 6.28319 | 0 | 3.14159 | 3 | 300 | 0.000623667 | 55531.9 | 0 | 1 |

Read row **ID 2** as: nitrogen gas between 9556.5 and 9558.5 km from the centre, or **0–2 km altitude**. Its pressure and density were calculated at the 1 km midpoint. ID 3 is the next atmospheric layer, at 2–4 km altitude.

The angular bounds `0 … 2π` in azimuth (φ) and `0 … π` in polar angle (θ) make every row a complete spherical shell. `Solid = 1` selects the solid material; `Solid = 0` selects the gas mixture. IDs label the layers sequentially.

### Fill the table with NumPy

`psf[2:-1]` selects all atmospheric rows. Assigning arrays to its columns fills all layers at once. Column indices start at zero: radii are columns 0–1, angles 2–5, ID 6, temperature 7, density 8, pressure 9, solid flag 10, and nitrogen fraction 11.

Our calculations used metres and kg/m³. Convert to the PSF's **km** and **g/cm³** when filling the table below.


In [ ]:
radius_km = radius / 1000
radii_km = radius_km + edges / 1000
psf = np.zeros((n_layers + 3, 12))

# Full spheres, sequential IDs, and a common temperature.
psf[:, 3] = 2 * np.pi
psf[:, 5] = np.pi
psf[:, 6] = np.arange(n_layers + 3)
psf[:, 7] = temperature

# Atmospheric rows: inner/outer radius, density, pressure, nitrogen fraction.
psf[2:-1, 0] = radii_km[:-1]
psf[2:-1, 1] = radii_km[1:]
psf[2:-1, 8] = density / 1000  # kg/m³ -> g/cm³
psf[2:-1, 9] = pressure
psf[2:-1, 11] = 1.0           # pure elemental nitrogen by mass

# Interior, 20 m surface layer, and 20 m outer stopping shell.
solid_rows = np.array([0, 1, n_layers + 2])
psf[solid_rows, 0] = [0, radius_km - 0.02, radii_km[-1] + 0.001]
psf[solid_rows, 1] = [radius_km - 0.02, radius_km, radii_km[-1] + 0.021]
psf[solid_rows, 8] = 2.7       # solid density, g/cm³
psf[solid_rows, 9] = surface_pressure
psf[solid_rows, 10] = 1       # solid flag

output = Path("exoplanet-inputs/simulation.psf")
output.parent.mkdir(exist_ok=True)
formats = ["%.12e"] * 12
formats[6] = formats[10] = "%d"  # IDs and solid flags are integers
np.savetxt(output, psf, fmt=formats, header="O533Si467 N", comments="")
print(f"Saved {output.resolve()} ({psf.shape[0]} layers)")

The PSF is ready. Keep it as the primary input for your run. To turn it into geometry, run this in a terminal from the same working directory:

```bash
atris-psf-to-gdml exoplanet-inputs/simulation.psf
```

That produces `exoplanet-inputs/simulation.gdml`. From inside `exoplanet-inputs/`, continue with [phantom selection](../running-atris/prepare-simulation.md#optionally-select-phantoms), then prepare the particle inputs. Phantoms are optional.

For other gas mixtures, see [gas composition](../reference/inputs.md#layer-composition). The [PSF reference](../reference/inputs.md#planet-specification-format-psf) describes every column. The outer shell is a numerical stopping boundary, not part of the physical atmosphere.

**Next step:** [Select phantoms and prepare particle inputs](../running-atris/prepare-simulation.md#optionally-select-phantoms).
